In [12]:
# ==========================
# Imports
# ==========================
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd

# ==========================
# Config
# ==========================
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 25
DATA_DIR = './dataset'  # subfolders: cataract, diabetic_retinopathy, glaucoma, normal

# ==========================
# Read dataset
# ==========================
data = []
classes = sorted(os.listdir(DATA_DIR))
for cls in classes:
    cls_path = os.path.join(DATA_DIR, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        if os.path.isfile(img_path):
            data.append((img_path, cls))

df = pd.DataFrame(data, columns=['image_path', 'label'])
print(f"Total images: {len(df)}")
print(f"Classes found: {df['label'].unique()}")

# Encode labels
label_to_index = {label: idx for idx, label in enumerate(classes)}
df['label_idx'] = df['label'].map(label_to_index)
NUM_CLASSES = len(classes)

# Train/validation split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_idx'], random_state=42)
print(f"Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

# ==========================
# Data augmentation
# ==========================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# ==========================
# Dataset preprocessing
# ==========================
def preprocess_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def df_to_dataset(dataframe, shuffle=True, batch_size=BATCH_SIZE):
    paths = dataframe['image_path'].values
    labels = dataframe['label_idx'].values
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe))
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = df_to_dataset(train_df)
val_dataset = df_to_dataset(val_df, shuffle=False)

# ==========================
# Build the model using DenseNet121
# ==========================
base_model = tf.keras.applications.DenseNet121(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze base

inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='AUC')]
)

model.summary()

# ==========================
# Train
# ==========================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

# ==========================
# Fine-tuning (optional)
# ==========================
# Unfreeze last 50 layers
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='AUC')]
)

history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)


Total images: 4217
Classes found: ['cataract' 'diabetic_retinopathy' 'glaucoma' 'normal']
Train samples: 3373, Validation samples: 844
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step 


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_7 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │         4,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,041,604 (26.86 MB)

 Trainable params: 4,100 (16.02 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

Epoch 1/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 23s 88ms/step - AUC: 0.4956 - accuracy: 0.2446 - loss: 1.9082 - val_AUC: 0.6367 - val_accuracy: 0.3187 - val_loss: 1.3163
Epoch 2/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - AUC: 0.5829 - accuracy: 0.3223 - loss: 1.6279 - val_AUC: 0.7637 - val_accuracy: 0.4633 - val_loss: 1.1420
Epoch 3/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - AUC: 0.6721 - accuracy: 0.4079 - loss: 1.3967 - val_AUC: 0.8197 - val_accuracy: 0.5486 - val_loss: 1.0347
Epoch 4/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - AUC: 0.7143 - accuracy: 0.4474 - loss: 1.2964 - val_AUC: 0.8537 - val_accuracy: 0.6066 - val_loss: 0.9515
Epoch 5/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - AUC: 0.7607 - accuracy: 0.5061 - loss: 1.1827 - val_AUC: 0.8707 - val_accuracy: 0.6351 - val_loss: 0.9003
Epoch 6/25
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - AUC: 0.7887 - accuracy: 0.5366 - loss: 1.1090 - val_AUC: 0.8859 - val_accuracy: 0.6611 - val_loss: 0.8549
Epoch 7/25
106/106 ━━━━━━━━━

In [13]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9442 - accuracy: 0.7857 - loss: 0.5801 - val_AUC: 0.9631 - val_accuracy: 0.8270 - val_loss: 0.4628
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9456 - accuracy: 0.7833 - loss: 0.5683 - val_AUC: 0.9650 - val_accuracy: 0.8294 - val_loss: 0.4509
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9457 - accuracy: 0.7833 - loss: 0.5737 - val_AUC: 0.9650 - val_accuracy: 0.8341 - val_loss: 0.4482
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9477 - accuracy: 0.7919 - loss: 0.5556 - val_AUC: 0.9662 - val_accuracy: 0.8400 - val_loss: 0.4402
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9515 - accuracy: 0.8031 - loss: 0.5347 - val_AUC: 0.9673 - val_accuracy: 0.8483 - val_loss: 0.4316
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9507 - accuracy: 0.7978 - loss: 0.5394 - val_AUC: 0.9668 - val_accuracy: 0.8483 - val_loss: 0.4311
Epoch 7/10
106/106 ━━━━━━━━━

In [14]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=32
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9602 - accuracy: 0.8183 - loss: 0.4792 - val_AUC: 0.9721 - val_accuracy: 0.8673 - val_loss: 0.3935
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9584 - accuracy: 0.8189 - loss: 0.4917 - val_AUC: 0.9721 - val_accuracy: 0.8685 - val_loss: 0.3931
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9604 - accuracy: 0.8129 - loss: 0.4791 - val_AUC: 0.9724 - val_accuracy: 0.8720 - val_loss: 0.3914
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9602 - accuracy: 0.8260 - loss: 0.4795 - val_AUC: 0.9729 - val_accuracy: 0.8732 - val_loss: 0.3863
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9658 - accuracy: 0.8417 - loss: 0.4405 - val_AUC: 0.9734 - val_accuracy: 0.8744 - val_loss: 0.3806
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9651 - accuracy: 0.8343 - loss: 0.4459 - val_AUC: 0.9741 - val_accuracy: 0.8756 - val_loss: 0.3739
Epoch 7/10
106/106 ━━━━━━━━━

In [15]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9693 - accuracy: 0.8485 - loss: 0.4159 - val_AUC: 0.9754 - val_accuracy: 0.8886 - val_loss: 0.3608
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9664 - accuracy: 0.8411 - loss: 0.4360 - val_AUC: 0.9756 - val_accuracy: 0.8815 - val_loss: 0.3591
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9670 - accuracy: 0.8458 - loss: 0.4310 - val_AUC: 0.9758 - val_accuracy: 0.8863 - val_loss: 0.3576
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9696 - accuracy: 0.8444 - loss: 0.4159 - val_AUC: 0.9762 - val_accuracy: 0.8886 - val_loss: 0.3532
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9736 - accuracy: 0.8574 - loss: 0.3852 - val_AUC: 0.9762 - val_accuracy: 0.8851 - val_loss: 0.3534
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9722 - accuracy: 0.8497 - loss: 0.3947 - val_AUC: 0.9763 - val_accuracy: 0.8874 - val_loss: 0.3525
Epoch 7/10
106/106 ━━━━━━━━━

In [16]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9724 - accuracy: 0.8547 - loss: 0.3947 - val_AUC: 0.9779 - val_accuracy: 0.8969 - val_loss: 0.3366
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9725 - accuracy: 0.8592 - loss: 0.3902 - val_AUC: 0.9779 - val_accuracy: 0.8957 - val_loss: 0.3362
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9752 - accuracy: 0.8696 - loss: 0.3682 - val_AUC: 0.9784 - val_accuracy: 0.8981 - val_loss: 0.3316
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9721 - accuracy: 0.8485 - loss: 0.3957 - val_AUC: 0.9786 - val_accuracy: 0.8981 - val_loss: 0.3296
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9758 - accuracy: 0.8630 - loss: 0.3683 - val_AUC: 0.9792 - val_accuracy: 0.8981 - val_loss: 0.3240
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9737 - accuracy: 0.8589 - loss: 0.3858 - val_AUC: 0.9789 - val_accuracy: 0.9017 - val_loss: 0.3267
Epoch 7/10
106/106 ━━━━━━━━━

In [17]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - AUC: 0.9764 - accuracy: 0.8636 - loss: 0.3610 - val_AUC: 0.9796 - val_accuracy: 0.8981 - val_loss: 0.3219
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9796 - accuracy: 0.8793 - loss: 0.3307 - val_AUC: 0.9795 - val_accuracy: 0.8957 - val_loss: 0.3216
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9794 - accuracy: 0.8755 - loss: 0.3359 - val_AUC: 0.9798 - val_accuracy: 0.8957 - val_loss: 0.3195
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9784 - accuracy: 0.8752 - loss: 0.3429 - val_AUC: 0.9809 - val_accuracy: 0.8993 - val_loss: 0.3128
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9790 - accuracy: 0.8740 - loss: 0.3359 - val_AUC: 0.9803 - val_accuracy: 0.8981 - val_loss: 0.3150
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9799 - accuracy: 0.8767 - loss: 0.3323 - val_AUC: 0.9799 - val_accuracy: 0.8981 - val_loss: 0.3155
Epoch 7/10
106/106 ━━━━━━━━━

In [18]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9815 - accuracy: 0.8838 - loss: 0.3158 - val_AUC: 0.9808 - val_accuracy: 0.8993 - val_loss: 0.3126
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9833 - accuracy: 0.8894 - loss: 0.2995 - val_AUC: 0.9804 - val_accuracy: 0.9028 - val_loss: 0.3132
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9832 - accuracy: 0.8823 - loss: 0.3026 - val_AUC: 0.9808 - val_accuracy: 0.8981 - val_loss: 0.3091
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - AUC: 0.9800 - accuracy: 0.8787 - loss: 0.3267 - val_AUC: 0.9810 - val_accuracy: 0.9017 - val_loss: 0.3068
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9822 - accuracy: 0.8784 - loss: 0.3098 - val_AUC: 0.9812 - val_accuracy: 0.9052 - val_loss: 0.3055
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9830 - accuracy: 0.8873 - loss: 0.3038 - val_AUC: 0.9815 - val_accuracy: 0.9028 - val_loss: 0.3028
Epoch 7/10
106/106 ━━━━━━━━━

In [19]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9835 - accuracy: 0.8897 - loss: 0.2975 - val_AUC: 0.9810 - val_accuracy: 0.9028 - val_loss: 0.3096
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9838 - accuracy: 0.8897 - loss: 0.2954 - val_AUC: 0.9808 - val_accuracy: 0.9040 - val_loss: 0.3134
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9848 - accuracy: 0.8953 - loss: 0.2860 - val_AUC: 0.9813 - val_accuracy: 0.9028 - val_loss: 0.3090
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9855 - accuracy: 0.8933 - loss: 0.2787 - val_AUC: 0.9818 - val_accuracy: 0.9028 - val_loss: 0.3034
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9836 - accuracy: 0.8950 - loss: 0.2963 - val_AUC: 0.9817 - val_accuracy: 0.9017 - val_loss: 0.3046
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9844 - accuracy: 0.8906 - loss: 0.2900 - val_AUC: 0.9818 - val_accuracy: 0.9017 - val_loss: 0.3042
Epoch 7/10
106/106 ━━━━━━━━━

In [20]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9855 - accuracy: 0.8995 - loss: 0.2712 - val_AUC: 0.9822 - val_accuracy: 0.9005 - val_loss: 0.3037
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9858 - accuracy: 0.8950 - loss: 0.2715 - val_AUC: 0.9820 - val_accuracy: 0.9040 - val_loss: 0.3012
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9853 - accuracy: 0.8962 - loss: 0.2801 - val_AUC: 0.9825 - val_accuracy: 0.9028 - val_loss: 0.2994
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9847 - accuracy: 0.8962 - loss: 0.2827 - val_AUC: 0.9818 - val_accuracy: 0.9017 - val_loss: 0.3031
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9860 - accuracy: 0.8986 - loss: 0.2735 - val_AUC: 0.9825 - val_accuracy: 0.9005 - val_loss: 0.3011
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - AUC: 0.9870 - accuracy: 0.9045 - loss: 0.2629 - val_AUC: 0.9822 - val_accuracy: 0.9017 - val_loss: 0.3002
Epoch 7/10


KeyboardInterrupt: 

In [21]:
model.save('eye_disease_classification.keras')

In [1]:
import tensorflow as tf

2025-10-14 19:13:10.499116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model2=tf.keras.models.load_model('eye_disease_classification.keras')

I0000 00:00:1760461992.865466  184600 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8244 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:06:00.0, compute capability: 8.6


In [16]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import os

img_path = "image_2025-10-14_20-25-07.png"            # Change to your test image
img_size = (224, 224)            # Must match the size used in training

# --- CLASS NAMES (match training folders) ---
class_names = sorted(os.listdir("dataset"))
print("Classes:", class_names)

# --- LOAD & PREPROCESS IMAGE ---
img = image.load_img(img_path, target_size=img_size)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
img_array = img_array / 255.0  # Normalize same as training

# --- PREDICT ---
pred = model2.predict(img_array)
pred_class = np.argmax(pred, axis=1)[0]
pred_label = class_names[pred_class]

print(f"✅ Predicted Class: {pred_label}")


Classes: ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
✅ Predicted Class: normal
